# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [3]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [4]:
# TODO
# Create a .env file in this folder (project/starter) with the following variables
# (copy .env.example from the repo root):
# OPENAI_API_KEY="voc-..."
# OPENAI_BASE_URL="https://openai.vocareum.com/v1"
# TAVILY_API_KEY="tvly-..."

### VectorDB Instance

In [5]:
# Instantiate your ChromaDB Client
# Keep this path: Part 2 loads the database from "chromadb" (inside project/starter)
chroma_client = chromadb.PersistentClient(path="chromadb")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


### Collection

In [6]:
# Pick one embedding function
# Part 2 rebuilds this same function when it loads the collection,
# so if you pick something different, keep it identical in both notebooks.
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

In [7]:
# Create a collection
# Keep this name: Part 2 loads the collection "udaplay"
collection = chroma_client.get_or_create_collection(
   name="udaplay",
   embedding_function=embedding_fn
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Add documents

In [8]:
# Run this notebook from project/starter: the game files are in ./games
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # Create content string for embedding
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use filename (without extension) as document ID
    doc_id = os.path.splitext(file_name)[0]
    
    # Add to collection
    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[{**game, "Source": file_name}]
    )

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


In [9]:
# Display collection statistics
print("=== ChromaDB Collection Statistics ===")
print(f"Collection name: {collection.name}")
print(f"Total documents: {collection.count()}")

=== ChromaDB Collection Statistics ===
Collection name: udaplay
Total documents: 15


### Semantic Search Demo
Cool, we have loaded all the game data into our vector database, let's demonstrate that semantic search works properly.

In [10]:
def test_semantic_search(collection, query: str, n_results: int = 5):
    """
    Runs a semantic search query against a ChromaDB collection and returns
    a structured list of results containing distance, metadata, and document text.
    """

    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    print(f"Query: {query}\nResult:")
    
    if results['documents'][0]:
        for i, (doc, meta, dist) in enumerate(zip(
                results['documents'][0], 
                results['metadatas'][0], 
                results['distances'][0]
            )):
            similarity = 1 - dist

            print(f"{i+1}. {meta.get('Name')} ({meta.get('YearOfRelease')})")
            print(f"   Similarity: {similarity:.3f}")
            print("   Platform:", meta.get("Platform"))
            print("   Genre:", meta.get("Genre"))
            print("   Source:", meta.get("Source"))
            print("   Description:", meta.get("Description"))
    else:
        print("No results found.")

In [11]:
# Test with different types of queries
test_queries = [
    "Realistic racing simulator on PlayStation 1",
    "Nintendo racing games",
    "RPG games with fantasy themes",
    "PlayStation games from the 1990s",
    "Action games with shooting mechanics",
    "Games suitable for families"
]

for query in test_queries:
    test_semantic_search(collection, query, n_results=2)
    print("\n")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: Realistic racing simulator on PlayStation 1
Result:
1. Gran Turismo (1997)
   Similarity: 0.831
   Platform: PlayStation 1
   Genre: Racing
   Source: 001.json
   Description: A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.
2. Gran Turismo 5 (2010)
   Similarity: 0.747
   Platform: PlayStation 3
   Genre: Racing
   Source: 003.json
   Description: A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.


Query: Nintendo racing games
Result:
1. Mario Kart 8 Deluxe (2017)
   Similarity: 0.708
   Platform: Nintendo Switch
   Genre: Racing
   Source: 012.json
   Description: An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.
2. Gran Turismo 5 (2010)
   Similarity: 0.655
   Platform: PlayStation 3
   Genre: Racing
   Source: 003.json
   Description: A comprehensive racing simulator featuring a vast selection of 